# ENNx CUDA-Oxide development on Colab

This notebook is the hosted T4 compatibility check. Select a GPU runtime before running the cells.

In [ ]:
import json
import platform
import subprocess
import sys

gpu = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name,compute_cap,driver_version,memory.total",
        "--format=csv,noheader",
    ],
    text=True,
).strip()
runtime = {
    "platform": platform.platform(),
    "python": platform.python_version(),
    "gpu": gpu,
}
print(json.dumps(runtime, indent=2))
assert sys.platform == "linux" and platform.machine() == "x86_64"
assert "T4" in gpu, f"Expected a T4 runtime, received: {gpu}"

In [ ]:
from pathlib import Path
import subprocess

ENN_REPO = "https://github.com/Kvutza/ennx.git"
ENN_REV = "main"
checkout = Path("/content/ennx")
if not checkout.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", ENN_REV, ENN_REPO, str(checkout)],
        check=True,
    )
print(checkout)

In [ ]:
subprocess.run(
    [sys.executable, "ops/colab_cuda_oxide_smoke.py", "setup"],
    cwd=checkout,
    check=True,
)

In [ ]:
subprocess.run(
    [sys.executable, "ops/colab_cuda_oxide_smoke.py", "doctor"],
    cwd=checkout,
    check=True,
)

In [ ]:
subprocess.run(
    [sys.executable, "ops/colab_cuda_oxide_smoke.py", "vecadd"],
    cwd=checkout,
    check=True,
)

A successful run ends with `SUCCESS: All 1024 elements correct!`. Next, compile the ENNx kernel for the T4 and compare it with the CPU implementation.

In [ ]:
subprocess.run(
    [sys.executable, "ops/colab_cuda_oxide_smoke.py", "ennx"],
    cwd=checkout,
    check=True,
)

In [ ]:
subprocess.run(
    [sys.executable, "ops/colab_cuda_oxide_smoke.py", "sanitize"],
    cwd=checkout,
    check=True,
)

In [ ]:
subprocess.run(
    [sys.executable, "ops/colab_cuda_oxide_smoke.py", "bench"],
    cwd=checkout,
    check=True,
)

Finally, compile the Python 3.12 extension and exercise the same resident trial engine through ENNx's public experimental API.

In [ ]:
subprocess.run(
    [sys.executable, "ops/colab_cuda_oxide_smoke.py", "python"],
    cwd=checkout,
    check=True,
)

The ENNx parity result must report 36 passing cases, Compute Sanitizer must report zero errors, and the Python check must report `CUDA_PYTHON ok=true`. Kernel work lives in `/content/ennx/cuda`.